In [ ]:
#storing csv files in dataframes

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df1 = pd.read_csv("C:\\Users\\Sunit\\OneDrive - Virginia Tech\\Documents\\Erdos Data science\\weather and health project\\fall-2025-extreme-temperatures-and-public-health\\Data\\Weather_data\\WeatherData\\london 1981-01-01 to 2019-12-31.csv")


df2 = pd.read_csv("C:\\Users\\Sunit\\OneDrive - Virginia Tech\\Documents\\Erdos Data science\\weather and health project\\fall-2025-extreme-temperatures-and-public-health\\Data\\data visualization\\deaths_region.csv")



In [ ]:
df_deaths = df2['Yorkshire & The Humber']  # <---region's deaths column
df_temp   = df1['temp']  # <---temperature column

len_deaths = len(df_deaths)
len_temp = len(df_temp)
print(f"Original lengths: Deaths={len_deaths}, Temp={len_temp}")

#one of the csv is only until 2014

if len_deaths != len_temp:
        data_length = min(len_deaths, len_temp)
        print(f"WARNING: Series have different lengths. Truncating both to {data_length} rows.")
        # Truncate both Series to the shorter length
        df_deaths = df_deaths.iloc[:data_length]
        df_temp = df_temp.iloc[:data_length]
else:
        data_length = len_deaths
        print("Series have the same length. Proceeding.")


Original lengths: Deaths=12783, Temp=14244


In [ ]:
import pandas as pd
import numpy as np

# --- 1. Align Series ---
df_deaths = df2['Yorkshire & The Humber']  # <--- respective deaths Series
df_temp   = df1['temp']    # <--- temperature Series

len_deaths = len(df_deaths)
len_temp = len(df_temp)
print(f"Original lengths: Deaths={len_deaths}, Temp={len_temp}")

if len_deaths != len_temp:
        data_length = min(len_deaths, len_temp)
        print(f"WARNING: Series have different lengths. Truncating both to {data_length} rows.")
        # Truncate both Series to the shorter length
        df_deaths = df_deaths.iloc[:data_length]
        df_temp = df_temp.iloc[:data_length]
else:
        data_length = len_deaths
        print("Series have the same length. Proceeding.")
        
# START DATE
START_DATE = "1985-01-01"  

# --- 2. Creating Date Index ---
try:
    date_index = pd.date_range(start=START_DATE, periods=data_length, freq='D')
    df_deaths.index = date_index
    df_temp.index = date_index
except NameError:
    print("\nCRITICAL ERROR: 'df_deaths' or 'df_temp' is not defined.")
    print("Please make sure df1 and df2 are loaded and the column names are correct in step 1.")
except Exception as e:
    print(f"\nCRITICAL ERROR creating date index: {e}")
    raise e

# --- 3. Creating NEW DataFrame for the DLNM model in R ---
# Original df_deaths and df_temp Series will NOT be modified.
print(f"\nCreating temporary model DataFrame starting from {START_DATE}...")
df = pd.DataFrame({
    'deaths': df_deaths,
    'temp': df_temp
})



Original lengths: Deaths=12783, Temp=14244

Creating temporary model DataFrame starting from 1985-01-01...


In [ ]:
# --- 4. Clean and prepare the new DataFrame when missing some data ---
# Drop any days where we have no death data
df = df.dropna(subset=['deaths'])

# For missing temperature data, fill it by interpolating
df['temp'] = df['temp'].interpolate(method='time')

# If any NaNs remain (e.g., at the very start), drop them
df = df.dropna()

# --- 5. Create Time-Based Features ---
print("Engineering time-based features (time, doy, dow)...")
df['time'] = np.arange(len(df))
df['doy'] = df.index.dayofyear
df['dow'] = df.index.dayofweek

# --- 6. Final Model-Ready DataFrame (MODIFIED) ---
# We MUST extract the date index into a named column before export.
df_model_ready = df[['deaths', 'temp', 'time', 'doy', 'dow']].rename(
    columns={'temp': 'temp_0'}
)

# *** MODIFICATION START ***
# 6a. Extract the datetime index into a new, named column 'date_col'
df_model_ready['date_col'] = df_model_ready.index

# 6b. Reorder columns to put the date first (good practice)
df_model_ready = df_model_ready[['date_col', 'deaths', 'temp_0', 'time', 'doy', 'dow']]
# *** MODIFICATION END ***

print(f"\nTemporary model data is ready with {len(df_model_ready)} observations.")
print("Columns:", df_model_ready.columns.to_list())
df_model_ready.head()

# --- 7. Export the Final DataFrame to CSV (MODIFIED) ---
# We use index=False because we explicitly saved the index as 'date_col'
df_model_ready.to_csv("Y_and_H_data.csv", index=False)

print("\nData exported to london_data.csv WITHOUT the DataFrame index.")
print("The date column is now explicitly named 'date_col'. You are ready for R.")

Engineering time-based features (time, doy, dow)...

Temporary model data is ready with 12783 observations.
Columns: ['date_col', 'deaths', 'temp_0', 'time', 'doy', 'dow']

Data exported to london_data.csv WITHOUT the DataFrame index.
The date column is now explicitly named 'date_col'. You are ready for R.


In [1]:
import os
print(os.getcwd())

c:\Users\Sunit\OneDrive - Virginia Tech\Documents\Erdos Data science\weather and health project\fall-2025-extreme-temperatures-and-public-health\Data


In [13]:
df_model_ready

,date_col,deaths,temp_0,time,doy,dow
1985-01-01,1985-01-01,238,6.3,0,1,1
1985-01-02,1985-01-02,202,8.9,1,2,2
1985-01-03,1985-01-03,248,9.6,2,3,3
1985-01-04,1985-01-04,228,4.1,3,4,4
1985-01-05,1985-01-05,253,1.7,4,5,5
...,...,...,...,...,...,...
2019-12-27,2019-12-27,141,14.0,12778,361,4
2019-12-28,2019-12-28,149,11.7,12779,362,5
2019-12-29,2019-12-29,138,11.1,12780,363,6
2019-12-30,2019-12-30,131,12.2,12781,364,0


In [15]:
import pandas as pd
import io

# Assume your last export was: df_model_ready.to_csv("london_data.csv", index=False)

# Read the file back in, telling pandas to look at the 'date_col' column as a date
df_check = pd.read_csv(
    "london_data.csv", 
    parse_dates=['date_col'] # Tell pandas this column *should* be a datetime
)

# Check the data type of the column
print(df_check['date_col'].dtype)

# Check the first few values to ensure the format is correct
print(df_check['date_col'])

datetime64[ns]
0       1985-01-01
1       1985-01-02
2       1985-01-03
3       1985-01-04
4       1985-01-05
           ...    
12778   2019-12-27
12779   2019-12-28
12780   2019-12-29
12781   2019-12-30
12782   2019-12-31
Name: date_col, Length: 12783, dtype: datetime64[ns]
